1.
<p>Create 'readings_messy.csv' from your readings by blanking 5% of `units_kwh`, turning `amount_inr` into text like 'Rs 1234.50' and mixing the case of `paid` (yes, YES, No). Load it, run `df.info()` and `df.describe(include='all')`, then build `issue_report` with one row per column (dtype, missing_count, unique_count) saved as 'society_issue_report.csv'. Run Restart & Run All and push to 's04_society_collect' in your GitHub repo 'ds_analysis_portfolio'. Constraint: Only detect the problems; do not fix them yet.</p>
2.
<p>Generate 1,200 monthly electricity readings for a fictional housing society in a notebook named 'society_collect.ipynb' with `np.random.seed(42)`: flat_id (A-01 to D-25), month (2025-01 to 2025-12), units_kwh 60 to 420, amount_inr = units_kwh x 7.5, paid (90% Yes). Save it as 'readings_comma.csv', 'readings_semicolon.csv' (`sep=';'`), 'readings_tab.tsv' (`sep='\t'`), 'readings.xlsx' and 'readings.json' (`orient='records'`). Hint: Use `index=False`.</p>
3.
<p>Load all five files back into DataFrames with `pd.read_csv` using the right `sep`, `pd.read_excel` and `pd.read_json`. Print a table of file name, shape and the `units_kwh` sum for each, and a final line that says whether all five match. Hint: `pd.read_excel` needs `openpyxl`, which comes with Anaconda.</p>
4.
<p>Generate 'meter_big.csv' with 60,000 hourly meter rows (flat_id, hour_no, kwh between 0.1 and 2.5, rounded to 2 decimals). Read it with `pd.read_csv(chunksize=10000)`, loop over the chunks to total the rows and `kwh` without loading the whole file, and print the chunk count (6) and both totals against the totals of the generating DataFrame, which must match within 0.01. Hint: Each chunk is a DataFrame, so use `chunk['kwh'].sum()` inside the loop.</p>

In [2]:
import numpy as np
import pandas as pd

In [3]:
np.random.seed(42)

# Generate 100 flat IDs: A-01 to D-25
flat_ids = []

for block in ["A", "B", "C", "D"]:
    for number in range(1, 26):
        flat_ids.append(f"{block}-{number:02d}")

# Generate 12 months
months = pd.date_range("2025-01-01", periods=12, freq="MS")
month_names = months.strftime("%Y-%m").tolist()

# Generate combinations of flats and months
flats = np.repeat(flat_ids, 12)
month_values = np.tile(month_names, 100)

# Generate electricity units
units_kwh = np.random.randint(60, 421, size=1200)

# Calculate amount
amount_inr = units_kwh * 7.5

# Generate payment status (90% Yes, 10% No)
paid = np.random.choice(
    ["Yes", "No"],
    size=1200,
    p=[0.90, 0.10]
)

# Create DataFrame
readings = pd.DataFrame({
    "flat_id": flats,
    "month": month_values,
    "units_kwh": units_kwh,
    "amount_inr": amount_inr,
    "paid": paid
})

print(readings.head())
print("\nTotal rows:", len(readings))
print("Total columns:", len(readings.columns))

  flat_id    month  units_kwh  amount_inr paid
0    A-01  2025-01        162      1215.0  Yes
1    A-01  2025-02        408      3060.0  Yes
2    A-01  2025-03        330      2475.0  Yes
3    A-01  2025-04        166      1245.0  Yes
4    A-01  2025-05        131       982.5  Yes

Total rows: 1200
Total columns: 5


In [4]:
# CSV with comma separator
readings.to_csv("readings_comma.csv", index=False)

# CSV with semicolon separator
readings.to_csv("readings_semicolon.csv", sep=";", index=False)

# TSV with tab separator
readings.to_csv("readings_tab.tsv", sep="\t", index=False)

# Excel file
readings.to_excel("readings.xlsx", index=False)

# JSON file
readings.to_json("readings.json", orient="records")

print("All five files saved successfully!")

All five files saved successfully!


In [5]:
df_comma = pd.read_csv("readings_comma.csv")

df_semicolon = pd.read_csv(
    "readings_semicolon.csv",
    sep=";"
)

df_tab = pd.read_csv(
    "readings_tab.tsv",
    sep="\t"
)

df_excel = pd.read_excel("readings.xlsx")

df_json = pd.read_json("readings.json")

print("All five files loaded successfully!")

All five files loaded successfully!


In [6]:
file_data = {
    "readings_comma.csv": df_comma,
    "readings_semicolon.csv": df_semicolon,
    "readings_tab.tsv": df_tab,
    "readings.xlsx": df_excel,
    "readings.json": df_json
}

comparison = []

for filename, df in file_data.items():
    comparison.append({
        "File Name": filename,
        "Shape": df.shape,
        "Units Sum": df["units_kwh"].sum()
    })

comparison_df = pd.DataFrame(comparison)

print(comparison_df)

# Check whether all five files match
all_match = (
    all(df.shape == readings.shape for df in file_data.values())
    and all(
        df["units_kwh"].sum() == readings["units_kwh"].sum()
        for df in file_data.values()
    )
)

print("\nDo all five files match?", all_match)

                File Name      Shape  Units Sum
0      readings_comma.csv  (1200, 5)     289085
1  readings_semicolon.csv  (1200, 5)     289085
2        readings_tab.tsv  (1200, 5)     289085
3           readings.xlsx  (1200, 5)     289085
4           readings.json  (1200, 5)     289085

Do all five files match? True


In [7]:
messy = readings.copy()

# Blank 5% of units_kwh (60 rows)
missing_rows = messy.sample(
    frac=0.05,
    random_state=42
).index

messy.loc[missing_rows, "units_kwh"] = np.nan

# Convert amount into text like Rs 1234.50
messy["amount_inr"] = messy["amount_inr"].apply(
    lambda x: f"Rs {x:.2f}"
)

# Mix the case of paid
messy["paid"] = np.random.choice(
    ["yes", "YES", "No"],
    size=len(messy)
)

# Save messy data
messy.to_csv("readings_messy.csv", index=False)

print("Messy CSV created successfully!")
print(messy.head())

Messy CSV created successfully!
  flat_id    month  units_kwh  amount_inr paid
0    A-01  2025-01      162.0  Rs 1215.00  yes
1    A-01  2025-02      408.0  Rs 3060.00  YES
2    A-01  2025-03      330.0  Rs 2475.00  YES
3    A-01  2025-04      166.0  Rs 1245.00  YES
4    A-01  2025-05      131.0   Rs 982.50  YES


In [8]:
df = pd.read_csv("readings_messy.csv")

print("First five rows:")
print(df.head())

print("\nDataFrame Information:")
df.info()

First five rows:
  flat_id    month  units_kwh  amount_inr paid
0    A-01  2025-01      162.0  Rs 1215.00  yes
1    A-01  2025-02      408.0  Rs 3060.00  YES
2    A-01  2025-03      330.0  Rs 2475.00  YES
3    A-01  2025-04      166.0  Rs 1245.00  YES
4    A-01  2025-05      131.0   Rs 982.50  YES

DataFrame Information:
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   flat_id     1200 non-null   str    
 1   month       1200 non-null   str    
 2   units_kwh   1140 non-null   float64
 3   amount_inr  1200 non-null   str    
 4   paid        1200 non-null   str    
dtypes: float64(1), str(4)
memory usage: 47.0 KB


In [9]:
print(df.describe(include="all"))

       flat_id    month    units_kwh  amount_inr  paid
count     1200     1200  1140.000000        1200  1200
unique     100       12          NaN         344     3
top       A-01  2025-01          NaN  Rs 3037.50   yes
freq        12      100          NaN           9   416
mean       NaN      NaN   240.893860         NaN   NaN
std        NaN      NaN   103.079724         NaN   NaN
min        NaN      NaN    60.000000         NaN   NaN
25%        NaN      NaN   157.750000         NaN   NaN
50%        NaN      NaN   238.000000         NaN   NaN
75%        NaN      NaN   330.000000         NaN   NaN
max        NaN      NaN   420.000000         NaN   NaN


In [10]:
issue_report = pd.DataFrame({
    "column": df.columns,
    "dtype": [df[col].dtype for col in df.columns],
    "missing_count": [df[col].isna().sum() for col in df.columns],
    "unique_count": [df[col].nunique() for col in df.columns]
})

print(issue_report)

       column    dtype  missing_count  unique_count
0     flat_id      str              0           100
1       month      str              0            12
2   units_kwh  float64             60           339
3  amount_inr      str              0           344
4        paid      str              0             3


In [11]:
issue_report.to_csv(
    "society_issue_report.csv",
    index=False
)

print("society_issue_report.csv saved successfully!")

society_issue_report.csv saved successfully!


In [12]:
np.random.seed(42)

# Generate 60,000 meter rows
meter_rows = 60000

meter_big = pd.DataFrame({
    "flat_id": np.random.choice(flat_ids, size=meter_rows),
    "hour_no": np.arange(1, meter_rows + 1),
    "kwh": np.round(
        np.random.uniform(0.1, 2.5, size=meter_rows),
        2
    )
})

print(meter_big.head())
print("\nTotal rows:", len(meter_big))

  flat_id  hour_no   kwh
0    C-02        1  0.60
1    D-18        2  2.10
2    A-15        3  1.25
3    C-22        4  0.65
4    C-11        5  0.99

Total rows: 60000


In [13]:
meter_big.to_csv("meter_big.csv", index=False)

print("meter_big.csv saved successfully!")

meter_big.csv saved successfully!


In [14]:
row_total = 0
kwh_total = 0
chunk_count = 0

for chunk in pd.read_csv("meter_big.csv", chunksize=10000):

    row_total += len(chunk)

    kwh_total += chunk["kwh"].sum()

    chunk_count += 1

print("Number of chunks:", chunk_count)
print("Total rows from chunks:", row_total)
print("Total kwh from chunks:", round(kwh_total, 2))

Number of chunks: 6
Total rows from chunks: 60000
Total kwh from chunks: 78216.07


In [15]:
original_rows = len(meter_big)
original_kwh = meter_big["kwh"].sum()

print("Original DataFrame rows:", original_rows)
print("Chunk processing rows:", row_total)

print("\nOriginal DataFrame kwh:", round(original_kwh, 2))
print("Chunk processing kwh:", round(kwh_total, 2))

# Check whether totals match
assert row_total == original_rows
assert abs(kwh_total - original_kwh) < 0.01

print("\nBoth row count and kwh totals match!")

Original DataFrame rows: 60000
Chunk processing rows: 60000

Original DataFrame kwh: 78216.07
Chunk processing kwh: 78216.07

Both row count and kwh totals match!
